In [1]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import requests
import os
from bs4 import BeautifulSoup
import re

import sys
sys.path.append('../src')
from data_loader import get_insideairbnb_url, download_city_data

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

In [2]:
import io

cities = {
    'los-angeles': 'united-states/ca/los-angeles',
    'new-york-city': 'united-states/ny/new-york-city',
    'chicago': 'united-states/il/chicago',
    'seattle': 'united-states/wa/seattle',
    'austin': 'united-states/tx/austin'
}

for city, path in cities.items():
    try:
        urls = get_insideairbnb_url(path, rank=1)
        response = requests.get(urls['listings'])
        df = pd.read_csv(io.BytesIO(response.content), compression='gzip', 
                        usecols=['price', 'estimated_occupancy_l365d', 'neighbourhood_cleansed'])
        print(f"\n{city}:")
        print(f"  Date: {urls['date']}")
        print(f"  Listings: {len(df)}")
        print(f"  Price non-null: {df['price'].notna().sum()}")
        print(f"  Occupancy non-null: {df['estimated_occupancy_l365d'].notna().sum()}")
        print(f"  Neighbourhoods: {df['neighbourhood_cleansed'].nunique()}")
    except Exception as e:
        print(f"\n{city}: FAILED — {e}")

Using date: 2025-09-01

los-angeles:
  Date: 2025-09-01
  Listings: 45886
  Price non-null: 36819
  Occupancy non-null: 45886
  Neighbourhoods: 266
Using date: 2025-11-01

new-york-city:
  Date: 2025-11-01
  Listings: 36353
  Price non-null: 21415
  Occupancy non-null: 36353
  Neighbourhoods: 224
Using date: 2025-06-17

chicago:
  Date: 2025-06-17
  Listings: 8604
  Price non-null: 7681
  Occupancy non-null: 8604
  Neighbourhoods: 76
Using date: 2025-06-21

seattle:
  Date: 2025-06-21
  Listings: 6862
  Price non-null: 6227
  Occupancy non-null: 6862
  Neighbourhoods: 88
Using date: 2025-06-13

austin:
  Date: 2025-06-13
  Listings: 15187
  Price non-null: 10708
  Occupancy non-null: 15187
  Neighbourhoods: 44


In [3]:
for city, path in cities.items():
    os.makedirs(f'../data/raw/{city}', exist_ok=True)
    urls = get_insideairbnb_url(path, rank=1)
    
    for name, url in urls.items():
        if name == 'date':
            continue
        ext = '.geojson' if 'geojson' in url else '.csv.gz'
        filepath = f'../data/raw/{city}/{name}{ext}'
        print(f'Downloading {city}/{name}...')
        response = requests.get(url)
        with open(filepath, 'wb') as f:
            f.write(response.content)

print("\nAll cities downloaded")

Using date: 2025-09-01
Using date: 2025-11-01
Using date: 2025-06-17
Using date: 2025-06-21
Using date: 2025-06-13

All cities downloaded


In [4]:
# Load all cities
dfs = []

for city in cities.keys():
    df = pd.read_csv(f'../data/raw/{city}/listings.csv.gz', compression='gzip')
    df['city'] = city  # add city label
    dfs.append(df)
    print(f"{city}: {len(df)} listings")

# Combine into one dataframe
listings = pd.concat(dfs, ignore_index=True)
print(f"\nTotal: {len(listings)} listings across {listings['city'].nunique()} cities")
print(f"Columns: {listings.shape[1]}")

los-angeles: 45886 listings
new-york-city: 36353 listings
chicago: 8604 listings
seattle: 6862 listings
austin: 15187 listings

Total: 112892 listings across 5 cities
Columns: 80


In [5]:
# Keep only relevant columns
keep_cols = [
    'city', 'id', 'neighbourhood_cleansed', 'latitude', 'longitude',
    'room_type', 'accommodates', 'bedrooms', 'price',
    'availability_365', 'estimated_occupancy_l365d', 'estimated_revenue_l365d',
    'number_of_reviews', 'review_scores_rating',
    'calculated_host_listings_count', 'license',
    'host_is_superhost', 'instant_bookable'
]

listings = listings[keep_cols].copy()

# Clean price column
listings['price_clean'] = (
    listings['price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .replace('nan', np.nan)
    .astype(float)
)

# Flag commercial hosts (more than 1 listing = likely not a regular person)
listings['is_commercial'] = listings['calculated_host_listings_count'] > 1

# Flag licensed listings
listings['is_licensed'] = listings['license'].notna()

print(listings.dtypes)
print(f"\nPrice non-null: {listings['price_clean'].notna().sum()}")
print(f"\nSample:\n{listings.head(3)}")

city                                  str
id                                  int64
neighbourhood_cleansed             object
latitude                          float64
longitude                         float64
room_type                             str
accommodates                        int64
bedrooms                          float64
price                                 str
availability_365                    int64
estimated_occupancy_l365d           int64
estimated_revenue_l365d           float64
number_of_reviews                   int64
review_scores_rating              float64
calculated_host_listings_count      int64
license                            object
host_is_superhost                     str
instant_bookable                      str
price_clean                       float64
is_commercial                        bool
is_licensed                          bool
dtype: object

Price non-null: 82850

Sample:
          city    id neighbourhood_cleansed  latitude  longitude  \
0  l

In [6]:
summary = listings.groupby('city').agg(
    total_listings=('id', 'count'),
    avg_occupancy=('estimated_occupancy_l365d', 'mean'),
    median_price=('price_clean', 'median'),
    commercial_pct=('is_commercial', 'mean'),
    licensed_pct=('is_licensed', 'mean'),
    avg_availability=('availability_365', 'mean')
).round(2)

summary['commercial_pct'] = (summary['commercial_pct'] * 100).round(1)
summary['licensed_pct'] = (summary['licensed_pct'] * 100).round(1)

print(summary.to_string())

               total_listings  avg_occupancy  median_price  commercial_pct  licensed_pct  avg_availability
city                                                                                                      
austin                  15187          62.14        138.00           54.00          0.00            174.44
chicago                  8604          87.42        169.00           69.00         69.00            220.58
los-angeles             45886          62.24        155.00           63.00         28.00            212.59
new-york-city           36353          46.98        154.00           51.00         15.00            165.38
seattle                  6862         108.43        177.00           61.00         83.00            198.73


## City-Level Summary Observations

- **Seattle** has the highest average occupancy (108 days/year), suggesting strong
  short-term rental demand relative to supply.
- **NYC** shows the lowest occupancy (47 days), likely reflecting strict STR regulations
  introduced in 2023 requiring host registration.
- **Austin** has 0% licensed listings — the city has no licensing requirement,
  making it the most permissive market in our sample.
- **Chicago** has the highest commercial host rate (69%), suggesting the market
  is dominated by property managers rather than individuals.

In [7]:
# Neighborhood-level aggregation
neighborhood = listings.groupby(['city', 'neighbourhood_cleansed']).agg(
    total_listings=('id', 'count'),
    avg_occupancy=('estimated_occupancy_l365d', 'mean'),
    median_price=('price_clean', 'median'),
    commercial_pct=('is_commercial', 'mean'),
    licensed_pct=('is_licensed', 'mean'),
    avg_availability=('availability_365', 'mean'),
    avg_reviews=('review_scores_rating', 'mean')
).round(2).reset_index()

# Occupancy rate as a fraction of year (0-1)
neighborhood['occupancy_rate'] = (neighborhood['avg_occupancy'] / 365).round(3)

# Airbnb activity score — combines listing density, occupancy, and commercialization
# We'll normalize each component to 0-1 then average them
for col in ['total_listings', 'avg_occupancy', 'commercial_pct']:
    min_val = neighborhood[col].min()
    max_val = neighborhood[col].max()
    neighborhood[f'{col}_norm'] = (neighborhood[col] - min_val) / (max_val - min_val)

neighborhood['activity_score'] = (
    neighborhood['total_listings_norm'] * 0.4 +
    neighborhood['avg_occupancy_norm'] * 0.4 +
    neighborhood['commercial_pct_norm'] * 0.2
).round(3)

print(f"Total neighborhoods: {len(neighborhood)}")
print(f"\nTop 10 by activity score:")
print(neighborhood.nlargest(10, 'activity_score')[
    ['city', 'neighbourhood_cleansed', 'total_listings', 
     'avg_occupancy', 'commercial_pct', 'activity_score']
].to_string())

Total neighborhoods: 698

Top 10 by activity score:
              city neighbourhood_cleansed  total_listings  avg_occupancy  commercial_pct  activity_score
399  new-york-city     Bedford-Stuyvesant            2580          50.08            0.49            0.58
3           austin                  78704            2284          63.67            0.51            0.56
517  new-york-city                Midtown            2037          48.27            0.82            0.56
252    los-angeles             Long Beach            1858          88.00            0.53            0.53
1           austin                  78702            1773          82.15            0.55            0.51
222    los-angeles              Hollywood            1816          42.19            0.69            0.49
506  new-york-city            Little Neck               4         214.00            0.75            0.49
371    los-angeles         West Hollywood            1220          71.31            0.75            0.45
314

## Neighborhood-Level Activity Score Observations

Computed an activity score per neighborhood combining listing density (40%),
average occupancy (40%), and commercial host rate (20%).

**Key findings:**
- **Bedford-Stuyvesant (NYC)** ranks highest — a historically Black neighborhood
  undergoing rapid gentrification, making high Airbnb activity here particularly
  significant from a housing displacement perspective.
- **Austin uses zip codes** instead of neighborhood names — a data quality issue
  to note when visualizing and communicating results.
- **Little Neck (NYC)** has only 4 listings but avg occupancy of 214 days —
  likely a statistical outlier driven by small sample size. Needs filtering.
- **Hollywood, Santa Monica, Venice** all appear in the top 10 for LA —
  consistent with their status as high-tourism neighborhoods.

**Limitation:** Activity score is normalized globally across all cities,
meaning large cities (NYC, LA) dominate by listing volume. 
Within-city normalization needed for fair cross-neighborhood comparison.

In [8]:
# Check suspicious high occupancy neighborhoods
outliers = neighborhood[neighborhood['avg_occupancy'] > 200]
print(outliers[['city', 'neighbourhood_cleansed', 'total_listings', 'avg_occupancy']])

              city neighbourhood_cleansed  total_listings  avg_occupancy
90         chicago        Mount Greenwood               1         255.00
444  new-york-city             Douglaston               3         205.00
506  new-york-city            Little Neck               4         214.00


In [9]:
# Filter neighborhoods with too few listings (unreliable averages)
MIN_LISTINGS = 10

neighborhood_filtered = neighborhood[neighborhood['total_listings'] >= MIN_LISTINGS].copy()

print(f"Before filter: {len(neighborhood)} neighborhoods")
print(f"After filter (n >= {MIN_LISTINGS}): {len(neighborhood_filtered)} neighborhoods")
print(f"\nRemoved: {len(neighborhood) - len(neighborhood_filtered)} neighborhoods")
print(f"\nMax occupancy after filter: {neighborhood_filtered['avg_occupancy'].max():.1f}")

Before filter: 698 neighborhoods
After filter (n >= 10): 596 neighborhoods

Removed: 102 neighborhoods

Max occupancy after filter: 149.7


In [10]:
# Recompute activity score normalized WITHIN each city
def normalize_within_city(df, col):
    return df.groupby('city')[col].transform(
        lambda x: (x - x.min()) / (x.max() - x.min())
    )

for col in ['total_listings', 'avg_occupancy', 'commercial_pct']:
    neighborhood_filtered[f'{col}_norm'] = normalize_within_city(
        neighborhood_filtered, col
    )

neighborhood_filtered['activity_score'] = (
    neighborhood_filtered['total_listings_norm'] * 0.4 +
    neighborhood_filtered['avg_occupancy_norm'] * 0.4 +
    neighborhood_filtered['commercial_pct_norm'] * 0.2
).round(3)

print("Top 3 neighborhoods by activity score per city:")
print(
    neighborhood_filtered
    .groupby('city')
    .apply(lambda x: x.nlargest(3, 'activity_score'), include_groups=False)
    .reset_index()
    [['city', 'neighbourhood_cleansed', 'total_listings', 
      'avg_occupancy', 'commercial_pct', 'activity_score']]
    .to_string()
)

Top 3 neighborhoods by activity score per city:
             city neighbourhood_cleansed  total_listings  avg_occupancy  commercial_pct  activity_score
0          austin                  78702            1773          82.15            0.55            0.78
1          austin                  78704            2284          63.67            0.51            0.71
2          austin                  78719              19          79.21            0.84            0.55
3         chicago              West Town             765         108.56            0.66            0.77
4         chicago        Near North Side            1038          64.05            0.91            0.76
5         chicago              Lake View             550         116.17            0.70            0.72
6     los-angeles             Long Beach            1858          88.00            0.53            0.75
7     los-angeles              Hollywood            1816          42.19            0.69            0.64
8     los-angele

## Neighborhood Activity Score — Within-City Normalized

Recomputed activity score normalized within each city to allow fair 
within-city comparison. Removed 102 neighborhoods with fewer than 10 listings.

**Top neighborhoods by city:**
- **Seattle:** Belltown (0.89), Broadway (0.84) — dense urban core dominates
- **Chicago:** West Town (0.77), Near North Side (0.76) — high commercial rate 
  in Near North Side (91%) stands out
- **Austin:** 78702, 78704 — zip codes, needs better labeling for communication
- **LA:** Long Beach (0.75), Hollywood (0.64)
- **NYC:** Bedford-Stuyvesant (0.62), Midtown (0.60)

In [11]:
# City-level ZORI
zori_url = "https://files.zillowstatic.com/research/public_csvs/zori/City_zori_uc_sfrcondomfr_sm_month.csv?t=1770952790"

print("Downloading city-level ZORI...")
response = requests.get(zori_url)
print(f"Status: {response.status_code}")

if response.status_code == 200:
    os.makedirs('../data/raw/zillow', exist_ok=True)
    with open('../data/raw/zillow/city_zori.csv', 'wb') as f:
        f.write(response.content)
    zori = pd.read_csv('../data/raw/zillow/city_zori.csv')
    print(f"Shape: {zori.shape}")
    print(zori.head(3))
else:
    print("Failed — trying metro level")

Status: 200
Shape: (4299, 143)
   RegionID  SizeRank   RegionName RegionType StateName State  \
0      6181         0     New York       city        NY    NY   
1     12447         1  Los Angeles       city        CA    CA   
2     39051         2      Houston       city        TX    TX   

                                   Metro          CountyName  2015-01-31  \
0  New York-Newark-Jersey City, NY-NJ-PA       Queens County     2564.80   
1     Los Angeles-Long Beach-Anaheim, CA  Los Angeles County     1742.04   
2   Houston-The Woodlands-Sugar Land, TX       Harris County     1168.00   

   2015-02-28  2015-03-31  2015-04-30  2015-05-31  2015-06-30  2015-07-31  \
0     2579.09     2600.71     2626.72     2648.59     2668.99     2680.36   
1     1754.75     1771.72     1783.40     1805.98     1815.84     1830.90   
2     1170.12     1174.49     1184.89     1195.99     1205.57     1204.96   

   2015-08-31  2015-09-30  2015-10-31  2015-11-30  2015-12-31  2016-01-31  \
0     2692.12    

In [12]:
# Identify date columns only (they start with 20)
date_cols = [c for c in zori.columns if c.startswith('20')]
id_cols = ['RegionID', 'RegionName', 'StateName', 'State', 'Metro']

# Melt only date columns
zori_long = zori[id_cols + date_cols].melt(
    id_vars=id_cols,
    var_name='date',
    value_name='rent'
)

# Filter to our 5 cities
city_map = {
    'Los Angeles': 'los-angeles',
    'New York': 'new-york-city',
    'Chicago': 'chicago',
    'Seattle': 'seattle',
    'Austin': 'austin'
}

zori_filtered = zori_long[zori_long['RegionName'].isin(city_map.keys())].copy()
zori_filtered['city'] = zori_filtered['RegionName'].map(city_map)
zori_filtered['date'] = pd.to_datetime(zori_filtered['date'], format='%Y-%m-%d')
zori_filtered = zori_filtered.dropna(subset=['rent'])

print(f"Shape: {zori_filtered.shape}")
print(zori_filtered.groupby('city')['date'].agg(['min', 'max']))
print(zori_filtered.head())

Shape: (707, 8)
                     min        max
city                               
austin        2015-01-31 2026-03-31
chicago       2015-01-31 2026-03-31
los-angeles   2015-01-31 2026-03-31
new-york-city 2015-01-31 2026-03-31
seattle       2015-01-31 2026-03-31
    RegionID   RegionName StateName State  \
0       6181     New York        NY    NY   
1      12447  Los Angeles        CA    CA   
3      17426      Chicago        IL    IL   
10     10221       Austin        TX    TX   
24     16037      Seattle        WA    WA   

                                    Metro       date    rent           city  
0   New York-Newark-Jersey City, NY-NJ-PA 2015-01-31 2564.80  new-york-city  
1      Los Angeles-Long Beach-Anaheim, CA 2015-01-31 1742.04    los-angeles  
3      Chicago-Naperville-Elgin, IL-IN-WI 2015-01-31 1511.01        chicago  
10       Austin-Round Rock-Georgetown, TX 2015-01-31 1140.58         austin  
24            Seattle-Tacoma-Bellevue, WA 2015-01-31 1505.34        sea

In [13]:
# Compute rent metrics per city
# 1. Latest rent (most recent month)
# 2. Rent 3 years ago (for growth calculation)
# 3. Rent growth over 3 years

latest_date = zori_filtered['date'].max()
three_years_ago = latest_date - pd.DateOffset(years=3)

# Get closest available dates
latest_rent = zori_filtered[zori_filtered['date'] == latest_date][['city', 'rent']].rename(columns={'rent': 'rent_latest'})
past_rent = zori_filtered[zori_filtered['date'] == zori_filtered[zori_filtered['date'] <= three_years_ago]['date'].max()][['city', 'rent']].rename(columns={'rent': 'rent_3yr_ago'})

city_rent = latest_rent.merge(past_rent, on='city')
city_rent['rent_growth_3yr_pct'] = ((city_rent['rent_latest'] - city_rent['rent_3yr_ago']) / city_rent['rent_3yr_ago'] * 100).round(2)

print(city_rent.sort_values('rent_growth_3yr_pct', ascending=False).to_string())

            city  rent_latest  rent_3yr_ago  rent_growth_3yr_pct
2        chicago      2327.75       1980.61                17.53
0  new-york-city      3810.95       3342.20                14.03
4        seattle      2193.37       2061.40                 6.40
1    los-angeles      2753.17       2644.05                 4.13
3         austin      1542.16       1715.59               -10.11
5         austin       969.03       1715.59               -43.52


In [14]:
# Check Austin duplicates
print(zori_filtered[zori_filtered['city'] == 'austin'].tail(10))

        RegionID RegionName StateName State                             Metro  \
558880     10221     Austin        TX    TX  Austin-Round Rock-Georgetown, TX   
560890     23555     Austin        MN    MN                        Austin, MN   
563179     10221     Austin        TX    TX  Austin-Round Rock-Georgetown, TX   
565189     23555     Austin        MN    MN                        Austin, MN   
567478     10221     Austin        TX    TX  Austin-Round Rock-Georgetown, TX   
569488     23555     Austin        MN    MN                        Austin, MN   
571777     10221     Austin        TX    TX  Austin-Round Rock-Georgetown, TX   
573787     23555     Austin        MN    MN                        Austin, MN   
576076     10221     Austin        TX    TX  Austin-Round Rock-Georgetown, TX   
578086     23555     Austin        MN    MN                        Austin, MN   

             date    rent    city  
558880 2025-11-30 1539.03  austin  
560890 2025-11-30  995.46  austin  


In [15]:
# Filter by state to avoid city name conflicts
state_map = {
    'Los Angeles': 'CA',
    'New York': 'NY',
    'Chicago': 'IL',
    'Seattle': 'WA',
    'Austin': 'TX'
}

# Apply both city name and state filter
mask = zori_long.apply(
    lambda row: row['RegionName'] in city_map and row['State'] == state_map[row['RegionName']], 
    axis=1
)
zori_filtered = zori_long[mask].copy()
zori_filtered['city'] = zori_filtered['RegionName'].map(city_map)
zori_filtered['date'] = pd.to_datetime(zori_filtered['date'], format='%Y-%m-%d')
zori_filtered = zori_filtered.dropna(subset=['rent'])

print(f"Shape: {zori_filtered.shape}")
print(zori_filtered.groupby('city')['date'].agg(['min', 'max', 'count']))

Shape: (674, 8)
                     min        max  count
city                                      
austin        2015-01-31 2026-03-31    135
chicago       2015-01-31 2026-03-31    135
los-angeles   2015-01-31 2026-03-31    135
new-york-city 2015-01-31 2026-03-31    134
seattle       2015-01-31 2026-03-31    135


In [16]:
latest_date = zori_filtered['date'].max()
three_years_ago = latest_date - pd.DateOffset(years=3)
past_date = zori_filtered[zori_filtered['date'] <= three_years_ago]['date'].max()

latest_rent = zori_filtered[zori_filtered['date'] == latest_date][['city', 'rent']].rename(columns={'rent': 'rent_latest'})
past_rent = zori_filtered[zori_filtered['date'] == past_date][['city', 'rent']].rename(columns={'rent': 'rent_3yr_ago'})

city_rent = latest_rent.merge(past_rent, on='city')
city_rent['rent_growth_3yr_pct'] = (
    (city_rent['rent_latest'] - city_rent['rent_3yr_ago']) / city_rent['rent_3yr_ago'] * 100
).round(2)

print(f"Comparing {past_date.date()} → {latest_date.date()}")
print(city_rent.sort_values('rent_growth_3yr_pct', ascending=False).to_string(index=False))

Comparing 2023-03-31 → 2026-03-31
         city  rent_latest  rent_3yr_ago  rent_growth_3yr_pct
      chicago      2327.75       1980.61                17.53
new-york-city      3810.95       3342.20                14.03
      seattle      2193.37       2061.40                 6.40
  los-angeles      2753.17       2644.05                 4.13
       austin      1542.16       1715.59               -10.11


In [17]:
# Merge rent growth with city-level Airbnb summary
city_summary = listings.groupby('city').agg(
    total_listings=('id', 'count'),
    avg_occupancy=('estimated_occupancy_l365d', 'mean'),
    median_price=('price_clean', 'median'),
    commercial_pct=('is_commercial', 'mean'),
    licensed_pct=('is_licensed', 'mean'),
).round(2).reset_index()

city_summary['commercial_pct'] = (city_summary['commercial_pct'] * 100).round(1)
city_summary['licensed_pct'] = (city_summary['licensed_pct'] * 100).round(1)

# Merge with rent data
city_analysis = city_summary.merge(city_rent, on='city')

print(city_analysis[['city', 'total_listings', 'avg_occupancy', 
                       'commercial_pct', 'rent_latest', 
                       'rent_growth_3yr_pct']].sort_values(
                           'rent_growth_3yr_pct', ascending=False
                       ).to_string(index=False))

         city  total_listings  avg_occupancy  commercial_pct  rent_latest  rent_growth_3yr_pct
      chicago            8604          87.42           69.00      2327.75                17.53
new-york-city           36353          46.98           51.00      3810.95                14.03
      seattle            6862         108.43           61.00      2193.37                 6.40
  los-angeles           45886          62.24           63.00      2753.17                 4.13
       austin           15187          62.14           54.00      1542.16               -10.11


In [18]:
# Save city-level analysis
city_analysis.to_csv('../data/processed/city_analysis.csv', index=False)

# Save neighborhood-level filtered data
neighborhood_filtered.to_csv('../data/processed/neighborhood_activity.csv', index=False)

# Save Zillow time series for plotting later
zori_filtered.to_csv('../data/processed/city_rent_timeseries.csv', index=False)

print("Saved all processed files")

Saved all processed files


## City-Level Airbnb + Rent Growth Merge

Merged InsideAirbnb city summaries with Zillow ZORI rent growth (2023→2026).

**Key finding:** No simple relationship between Airbnb activity and rent growth.
- Chicago: high occupancy (87 days) + high rent growth (17.5%)
- Austin: moderate occupancy (62 days) + falling rents (-10.1%)
- Austin's rent decline likely driven by housing supply boom, not Airbnb suppression

This complicates a naive "Airbnb = higher rents" narrative and points toward
a more nuanced story: Airbnb is one pressure among many on housing costs.